# Predicting Prolonged ICU Length of Stay Using First-Day Clinical Data

## Project Overview

This project uses the MIMIC-IV v3.1 critical care database to develop a machine learning model that predicts whether an ICU stay will exceed 5 days.

The prediction is based only on information available during the first 24 hours of ICU admission, including demographics, admission characteristics, vital signs, laboratory values, neurologic status, urine output, respiratory support, and vasoactive medication use.

The goal is to create a clinically meaningful prediction model while practicing real-world data extraction, SQL, exploratory data analysis, preprocessing, machine learning, model interpretation, and deployment.

## Data Source and Cohort Construction

The source data were obtained from the MIMIC-IV v3.1 database through Google BigQuery.

SQL was used to combine information from multiple relational tables, including:

- ICU stays
- Hospital admissions
- Patient demographics
- First-day vital signs
- First-day laboratory values
- First-day Glasgow Coma Scale
- First-day urine output
- Mechanical ventilation and respiratory support
- Vasoactive medication use

The analytic cohort was restricted to the first ICU stay within each hospital admission.

Patients who died during the hospitalization were excluded because an early death could otherwise be incorrectly classified as a short, non-prolonged ICU stay.

## Cohort Definition

This analysis uses a 24-hour landmark design.

Patients were included only if they remained in the ICU for at least 24 hours.
Clinical information collected during the first 24 hours of ICU admission was used
to predict whether the total ICU length of stay would exceed 5 days.

Patients with ICU stays shorter than 24 hours were excluded because they were no
longer at risk at the 24-hour prediction landmark.

The final cohort contains 59,809 ICU stays from 47,974 unique patients.

Outcome:
- 0 = ICU stay of 1–5 days
- 1 = ICU stay >5 days

The positive-class prevalence is approximately 19.5%.

## Outcome Definition

The target variable is:

**Prolonged ICU stay = ICU length of stay greater than 5 days**

The final 24-hour landmark cohort contains:

- 59,809 ICU stays
- 48,122 non-prolonged ICU stays
- 11,687 prolonged ICU stays
- 19.54% positive class prevalence

Each row represents one ICU stay.

## Initial Data Validation

Before beginning the exploratory analysis, the exported dataset was checked to confirm that:

- The expected number of ICU stays was preserved after export
- Each ICU stay appears only once
- The target distribution matches the SQL-derived cohort
- Column types and missing values were imported correctly

Identifier columns such as `subject_id`, `hadm_id`, and `stay_id` will not be used as model predictors.

The `los` variable is retained only for descriptive and validation purposes because it directly defines the target and would otherwise create target leakage.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/icu_prolonged_los_final_v2.csv")

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df["prolonged_icu_stay"].value_counts(normalize=True)

In [ ]:
df[["invasive_vent_24h", "invasive_vent_at_24h"]].value_counts().sort_index()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["prolonged_icu_stay"].value_counts(normalize=True)

In [ ]:
df["stay_id"].duplicated().sum()

### Initial Validation Results

Dataset contains 59,809 ICU stays and 45 columns.

There are no duplicated ICU stay identifiers.

Approximately 19.5% of ICU stays meet the definition of prolonged ICU stay (>5 days), producing an imbalanced binary classification problem.


# Descriptive Analysis
In this section, we summarize the main characteristics of the dataset before performing detailed exploratory analysis.

The goals are to:

- Identify the numerical and categorical predictors
- Summarize distributions using descriptive statistics
- Evaluate missing data
- Examine category frequencies
- Look for implausible values or extreme outliers
- Understand the degree of class imbalance in the target

No rows or variables will be removed at this stage. The goal is first to understand the data before making preprocessing decisions.

In [ ]:
id_cols = ["subject_id", "hadm_id", "stay_id"]
target_col = "prolonged_icu_stay"

categorical_cols = [
    "gender",
    "race",
    "admission_type",
    "admission_location",
    "first_careunit"
]

exclude_from_model = id_cols + [target_col, "los"]

numerical_cols = [
    col
    for col in df.columns
    if col not in categorical_cols + exclude_from_model
]

In [ ]:
print("Number of categorical predictors:", len(categorical_cols))
print("Number of numerical predictors:", len(numerical_cols))
print("Total predictors:", len(categorical_cols) + len(numerical_cols))

### Predictor Groups

The model predictors are divided into two broad groups:

- **Categorical predictors:** variables such as gender, race, admission type, admission location, and ICU type.
- **Numerical predictors:** age, vital signs, laboratory values, neurologic measures, urine output, and ICU-support variables.

The identifier columns (`subject_id`, `hadm_id`, and `stay_id`) are excluded because they identify encounters rather than describe patient physiology.

The `los` variable is also excluded because it directly determines the target variable and would cause target leakage.

In [ ]:
numerical_summary = df[numerical_cols].describe().T

numerical_summary

### Numerical Descriptive Statistics

The table above summarizes the central tendency and spread of each numerical predictor.

Important values to review include:

- **mean** and **median** to assess skewness
- **standard deviation** to assess variability
- **minimum and maximum** to identify potentially implausible values
- **25th and 75th percentiles** to understand the typical range

Clinical variables are often not normally distributed, particularly laboratory values and measures of illness severity, so large differences between the mean and median may indicate skewed distributions.

In [ ]:
for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

### Categorical Variable Distribution

Categorical predictors are reviewed to identify:

- dominant categories
- rare categories
- missing or unknown categories
- variables that may require grouping before one-hot encoding

Variables with many rare categories, particularly race and admission location, may later be simplified to reduce model complexity and unstable estimates.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df[numerical_cols + categorical_cols].isna().sum(),
    "missing_percent": (
        df[numerical_cols + categorical_cols]
        .isna()
        .mean()
        .mul(100)
    )
}).sort_values("missing_percent", ascending=False)

missing_summary

### Missing Data

Missing data are common in ICU datasets because laboratory tests and clinical measurements are obtained according to clinical need rather than according to a fixed research protocol.

Most predictors in the current analytic dataset have relatively low missingness.

The variables with the greatest missingness are:

- INR: approximately 9.3%
- Urine output: approximately 3.3%
- Temperature: approximately 2.7%

Most remaining variables have less than 1% missing data.

Rows were not removed simply because individual predictor values were missing. Missing numerical values are handled using median imputation within the machine-learning pipeline, while categorical variables use most-frequent imputation.

### Descriptive Analysis: Initial Findings

The dataset contains 40 candidate predictors: 35 numerical and 5 categorical variables.

Several observations are important before preprocessing:

1. Most physiological and laboratory variables have relatively little missing data.
2. INR has the highest missingness at approximately 9.3%.
3. Urine output is missing in approximately 3.3% of ICU stays.
4. Temperature measurements are missing in approximately 2.7% of ICU stays.
5. Several numerical variables contain extreme values that may represent true severe physiology, measurement error, charting error, or data-processing artifacts.
6. Some categorical variables contain relatively uncommon groups and should therefore be interpreted cautiously.

These findings guide the cleaning and preprocessing strategy used before model development.

### Potential Outliers and Data Quality

Several numerical predictors contain extreme values.

In critical-care datasets, extreme values can arise from:

- true severe physiology
- measurement artifacts
- unit errors
- documentation errors
- corrected chart entries
- aggregation of unusual source data

Extreme observations should therefore not automatically be removed. Their distributions will first be examined to distinguish plausible clinical extremes from clearly implausible values.

In [ ]:
quantiles = df[numerical_cols].quantile(
    [0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999]
).T

quantiles

In [ ]:
outlier_check_cols = [
    "heart_rate_min",
    "heart_rate_max",
    "sbp_min",
    "mbp_min",
    "temperature_min",
    "temperature_max",
    "spo2_min",
    "wbc_max",
    "creatinine_max",
    "glucose_max",
    "potassium_max",
    "urine_output_24h"
]

df[outlier_check_cols].describe(
    percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
).T

### Outlier Review

Extreme values were reviewed using percentile-based summaries rather than relying only on minimum and maximum values.

In ICU data, very abnormal measurements may represent genuine severe illness and should not automatically be removed. However, some values are physiologically implausible and are more likely to represent charting errors, measurement artifacts, or unit inconsistencies.

The preprocessing strategy will therefore:

- retain clinically plausible extreme values
- convert clearly implausible values to missing values
- impute those missing values later within the machine-learning pipeline
- avoid aggressive outlier removal that could discard clinically important information

In [ ]:
import numpy as np

clean_df = df.copy()

# Clearly implausible vital-sign values
clean_df.loc[clean_df["heart_rate_min"] < 20, "heart_rate_min"] = np.nan
clean_df.loc[clean_df["heart_rate_max"] > 250, "heart_rate_max"] = np.nan

clean_df.loc[clean_df["sbp_min"] < 30, "sbp_min"] = np.nan
clean_df.loc[clean_df["mbp_min"] < 20, "mbp_min"] = np.nan

clean_df.loc[clean_df["temperature_min"] < 25, "temperature_min"] = np.nan
clean_df.loc[clean_df["temperature_max"] > 45, "temperature_max"] = np.nan

clean_df.loc[clean_df["spo2_min"] < 20, "spo2_min"] = np.nan

# Implausibly high potassium
clean_df.loc[clean_df["potassium_max"] > 12, "potassium_max"] = np.nan

# Negative urine output is not physiologically meaningful
clean_df.loc[clean_df["urine_output_24h"] < 0, "urine_output_24h"] = np.nan

In [ ]:
checks = {
    "heart_rate_min < 20": (df["heart_rate_min"] < 20).sum(),
    "heart_rate_max > 250": (df["heart_rate_max"] > 250).sum(),
    "sbp_min < 30": (df["sbp_min"] < 30).sum(),
    "mbp_min < 20": (df["mbp_min"] < 20).sum(),
    "temperature_min < 25": (df["temperature_min"] < 25).sum(),
    "temperature_max > 45": (df["temperature_max"] > 45).sum(),
    "spo2_min < 20": (df["spo2_min"] < 20).sum(),
    "potassium_max > 12": (df["potassium_max"] > 12).sum(),
    "urine_output_24h < 0": (df["urine_output_24h"] < 0).sum(),
    "wbc_max > 100": (df["wbc_max"] > 100).sum(),
    "creatinine_max > 20": (df["creatinine_max"] > 20).sum(),
    "glucose_max > 1000": (df["glucose_max"] > 1000).sum()
}

checks

### Review of Extreme Values

Most extreme observations are uncommon.

Some values may represent true severe critical illness and should not be automatically removed. In particular, profound hypotension, severe leukocytosis, renal failure, marked hyperglycemia, and extreme heart rates can occur in critically ill patients.

A conservative cleaning strategy will therefore be used. Only values that are clearly physiologically implausible or incompatible with the measurement definition will initially be converted to missing values. Other extreme observations will be retained unless further inspection suggests that they are erroneous.

In [ ]:
df.loc[
    (df["sbp_min"] < 30),
    [
        "stay_id",
        "sbp_min",
        "sbp_mean",
        "mbp_min",
        "mbp_mean",
        "heart_rate_mean",
        "prolonged_icu_stay"
    ]
].sort_values("sbp_min").head(20)

In [ ]:
df.loc[
    (df["mbp_min"] < 20),
    [
        "stay_id",
        "sbp_min",
        "sbp_mean",
        "mbp_min",
        "mbp_mean",
        "heart_rate_mean",
        "prolonged_icu_stay"
    ]
].sort_values("mbp_min").head(20)

In [ ]:
df.loc[
    (df["potassium_max"] > 12)
    | (df["creatinine_max"] > 20)
    | (df["glucose_max"] > 1000)
    | (df["wbc_max"] > 100),
    [
        "stay_id",
        "potassium_max",
        "creatinine_max",
        "glucose_max",
        "wbc_max",
        "prolonged_icu_stay"
    ]
].sort_values("glucose_max", ascending=False).head(30)

### Physiologic Plausibility Cleaning

Inspection of extreme observations showed that several minimum vital-sign values were inconsistent with the remaining measurements from the same ICU stay.

For example, some ICU stays contained a systolic blood pressure minimum close to zero despite normal mean systolic and mean arterial pressures. These values are most consistent with isolated measurement or documentation artifacts.

A conservative cleaning strategy was therefore used:

- Only clearly implausible measurements were converted to missing values.
- Other extreme but clinically possible measurements were retained.
- Entire patient records were not removed because of an isolated implausible measurement.
- Missing values introduced during cleaning will later be handled within the preprocessing pipeline.

This approach preserves clinically meaningful severe physiology while reducing the influence of obvious measurement artifacts.

In [ ]:
cleaning_summary = pd.DataFrame({
    "original_missing": df.isna().sum(),
    "cleaned_missing": clean_df.isna().sum()
})

cleaning_summary["new_missing_from_cleaning"] = (
    cleaning_summary["cleaned_missing"]
    - cleaning_summary["original_missing"]
)

cleaning_summary[
    cleaning_summary["new_missing_from_cleaning"] > 0
].sort_values(
    "new_missing_from_cleaning",
    ascending=False
)

In [ ]:
clean_df.shape

# Exploratory Data Analysis

After reviewing data quality, missingness, and extreme values, the next step is to explore how the predictors relate to the target outcome.

The target variable is:

- `0`: ICU length of stay 1–5 days
- `1`: ICU length of stay > 5 days

The goals of this section are to:

- compare clinical characteristics between prolonged and non-prolonged ICU stays
- identify variables that may be associated with prolonged ICU stay
- examine categorical and ICU-support variables
- visualize important numerical predictors
- identify potential redundancy or correlation between predictors

These analyses are exploratory. Associations observed here do not establish causation.

In [ ]:
key_numeric_cols = [
    "age",
    "heart_rate_mean",
    "mbp_mean",
    "resp_rate_mean",
    "spo2_mean",
    "hemoglobin_min",
    "wbc_max",
    "bicarbonate_min",
    "bun_max",
    "creatinine_max",
    "glucose_max",
    "sodium_min",
    "potassium_max",
    "inr_max",
    "gcs_min",
    "urine_output_24h"
]

group_summary = (
    clean_df
    .groupby("prolonged_icu_stay")[key_numeric_cols]
    .agg(["median", "mean"])
    .T
)

group_summary

### Numerical Comparison by Outcome

Numerical variables were summarized separately for prolonged and non-prolonged ICU stays.

Both means and medians are shown. Medians are particularly useful for ICU variables because several clinical measurements have skewed distributions and extreme values.

Differences observed between the two groups will help identify candidate predictors, but these comparisons are descriptive rather than causal.

In [ ]:
support_cols = [
    "invasive_vent_24h",
    "invasive_vent_at_24h",
    "noninvasive_vent_24h",
    "hfnc_24h",
    "tracheostomy_24h",
    "any_vasoactive_24h"
]

support_rates = (
    clean_df
    .groupby("prolonged_icu_stay")[support_cols]
    .mean()
    .T
    .mul(100)
)

support_rates.columns = [
    "Non-prolonged (%)",
    "Prolonged (%)"
]

support_rates

### ICU Support During the First 24 Hours

The prevalence of major ICU therapies was compared between outcome groups.

These variables describe early illness severity and resource utilization, including:

- invasive mechanical ventilation
-invasive mechanical ventilation at the 24-hour landmark
- non-invasive ventilation
- high-flow nasal cannula
- tracheostomy
- vasoactive medication use

Because these variables are binary, their group means represent the proportion of ICU stays receiving each therapy.

In [ ]:
import matplotlib.pyplot as plt

support_rates.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylabel("Patients (%)")
plt.xlabel("ICU support variable")
plt.title("First-24-Hour ICU Support by Prolonged ICU Stay Status")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
clean_df.groupby("prolonged_icu_stay")[
    "vasoactive_agent_count_24h"
].describe()

In [ ]:
clean_df.boxplot(
    column="vasoactive_agent_count_24h",
    by="prolonged_icu_stay",
    figsize=(7, 5)
)

plt.title("Number of Vasoactive Agents by ICU Stay Outcome")
plt.suptitle("")
plt.xlabel("Prolonged ICU Stay")
plt.ylabel("Number of Vasoactive Agents")
plt.show()

In [ ]:
vent_table = pd.crosstab(
    clean_df["invasive_vent_24h"],
    clean_df["prolonged_icu_stay"],
    normalize="columns"
) * 100

vent_table

In [ ]:
icu_outcomes = (
    clean_df
    .groupby("first_careunit")
    .agg(
        total_stays=("prolonged_icu_stay", "size"),
        prolonged_stays=("prolonged_icu_stay", "sum"),
        prolonged_rate=("prolonged_icu_stay", "mean")
    )
    .sort_values("total_stays", ascending=False)
)

icu_outcomes["prolonged_rate"] *= 100

icu_outcomes

### ICU Type and Prolonged Stay

Prolonged ICU-stay rates were compared across the first ICU care unit.

Differences between ICUs should be interpreted primarily as differences in case mix and patient severity rather than as measures of ICU performance.

For example, medical, surgical, cardiovascular, trauma, and neurologic ICUs care for substantially different patient populations.

In [ ]:
plt.figure(figsize=(8, 5))

clean_df[
    clean_df["prolonged_icu_stay"] == 0
]["age"].hist(
    bins=30,
    alpha=0.6,
    density=True,
    label="≤5 days"
)

clean_df[
    clean_df["prolonged_icu_stay"] == 1
]["age"].hist(
    bins=30,
    alpha=0.6,
    density=True,
    label=">5 days"
)

plt.xlabel("Age")
plt.ylabel("Density")
plt.title("Age Distribution by ICU Length-of-Stay Group")
plt.legend()
plt.show()

### Key EDA Findings

Several clinically meaningful differences were observed between prolonged and non-prolonged ICU stays.

- Invasive mechanical ventilation during the first 24 hours was substantially more common among prolonged ICU stays.
- Vasoactive medication use was also more frequent in the prolonged-stay group.
- Prolonged stays were associated with modestly higher heart rate, respiratory rate, WBC count, BUN, creatinine, glucose, potassium, and INR.
- Prolonged stays showed somewhat lower hemoglobin, bicarbonate, GCS, and urine output.
- Age distributions overlapped substantially between outcome groups, suggesting that age alone is unlikely to strongly discriminate prolonged ICU stay.
- Prolonged-stay rates varied by ICU type, although these differences likely reflect differences in patient populations and case mix rather than ICU performance.
- Very small ICU categories produced unstable percentages and should be interpreted cautiously.

Overall, the exploratory findings suggest that early illness severity and intensity of organ support may be more informative than basic demographic characteristics for predicting prolonged ICU stay.

In [ ]:
important_continuous = [
    "heart_rate_mean",
    "mbp_mean",
    "creatinine_max",
    "bun_max",
    "gcs_min",
    "urine_output_24h"
]

for col in important_continuous:
    clean_df.boxplot(
        column=col,
        by="prolonged_icu_stay",
        figsize=(6, 4),
        showfliers=False
    )
    plt.title(f"{col} by Prolonged ICU Stay")
    plt.suptitle("")
    plt.xlabel("Prolonged ICU Stay")
    plt.ylabel(col)
    plt.show()

In [ ]:
correlation_cols = [
    "heart_rate_min",
    "heart_rate_max",
    "heart_rate_mean",
    "sbp_min",
    "sbp_mean",
    "mbp_min",
    "mbp_mean",
    "resp_rate_max",
    "resp_rate_mean",
    "temperature_min",
    "temperature_max",
    "spo2_min",
    "spo2_mean",
    "sodium_min",
    "sodium_max",
    "potassium_min",
    "potassium_max"
]

corr = clean_df[correlation_cols].corr()

plt.figure(figsize=(12, 9))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation_cols)), correlation_cols, rotation=90)
plt.yticks(range(len(correlation_cols)), correlation_cols)
plt.title("Correlation Among Related Physiologic Predictors")
plt.tight_layout()
plt.show()

### Interpretation of Continuous Predictors and Correlation

The boxplots show substantial overlap between prolonged and non-prolonged ICU stays for most individual physiological variables. This suggests that prolonged ICU stay is unlikely to be explained by a single measurement and instead reflects a combination of multiple clinical factors.

Several variables showed modest shifts in the prolonged-stay group, including higher heart rate, BUN, creatinine, and lower GCS and urine output.

The correlation matrix also identified groups of related predictors. For example, minimum, maximum, and mean heart rate measurements were strongly correlated, as were related blood-pressure measures and minimum/maximum electrolyte values.

Highly correlated predictors may contain redundant information. This is particularly relevant for regression-based models, where multicollinearity can make coefficient estimates unstable. Tree-based models are generally more tolerant of correlated predictors, although reducing redundancy may still improve interpretability.

### Feature Simplification

Several vital-sign variables were available as minimum, maximum, and mean values over the first ICU day.

Because these measures were highly correlated, the feature set was simplified to reduce redundancy and improve interpretability.

For continuously monitored physiological variables such as heart rate, mean arterial pressure, respiratory rate, and oxygen saturation, the mean first-day value was selected as the primary summary measure.

For laboratory variables, clinically relevant minimum or maximum values were retained when the more abnormal extreme was likely to better reflect illness severity.

## Train/Test Split

To avoid patient-level data leakage, the dataset is split by `subject_id` rather than by individual ICU stays.

This ensures that all ICU stays belonging to the same patient are assigned entirely to either the training set or the test set.

The target variable is `prolonged_icu_stay`, and the selected predictors include demographic, physiologic, laboratory, neurologic, renal, respiratory-support, and vasoactive-support variables.

In [ ]:
continuous_features = [
    "age",
    "heart_rate_mean",
    "mbp_mean",
    "resp_rate_mean",
    "spo2_mean",
    "hemoglobin_min",
    "platelets_min",
    "wbc_max",
    "bicarbonate_min",
    "bun_max",
    "creatinine_max",
    "glucose_max",
    "sodium_min",
    "potassium_max",
    "inr_max",
    "gcs_min",
    "urine_output_24h",
    "vasoactive_agent_count_24h"
]

binary_features = [
    "invasive_vent_24h",
    "invasive_vent_at_24h",
    "noninvasive_vent_24h",
    "hfnc_24h",
    "tracheostomy_24h",
    "any_vasoactive_24h"
]

final_categorical_features = [
    "gender",
    "admission_type",
    "admission_location",
    "first_careunit"
]

final_numeric_features = continuous_features + binary_features

feature_cols = (
    continuous_features
    + binary_features
    + final_categorical_features
)

print("Total predictors:", len(feature_cols))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = clean_df[feature_cols].copy()
y = clean_df["prolonged_icu_stay"].copy()
groups = clean_df["subject_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_subjects = clean_df.iloc[train_idx]["subject_id"]
test_subjects = clean_df.iloc[test_idx]["subject_id"]

In [ ]:
overlap = set(train_subjects).intersection(set(test_subjects))

print("Patient overlap:", len(overlap))
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

In [ ]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

### Train/Test Split Strategy

The dataset was split by patient identifier (`subject_id`) rather than by individual ICU stays.

This prevents different ICU stays from the same patient from appearing in both the training and test sets, reducing patient-level data leakage.

`GroupShuffleSplit` was used to reserve 20% of the cohort for final testing.

Because `GroupShuffleSplit` does not explicitly stratify the outcome, the class distribution in the training and test sets was checked after splitting.

There was no overlap in patient identifiers between the training and test sets.

## Preprocessing Pipeline

The modeling dataset contains both numerical and categorical predictors, as well as some missing values.

To avoid data leakage, all preprocessing steps are learned from the training set only and then applied unchanged to the test set.

The preprocessing strategy is:

- numerical variables: median imputation
- categorical variables: most-frequent imputation if needed, followed by one-hot encoding
- scaling of numerical variables for models that are sensitive to feature scale, such as logistic regression

The preprocessing steps will be combined using a `ColumnTransformer` so that the same transformations can later be incorporated directly into the machine-learning pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

In [ ]:
binary_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", numeric_pipeline, continuous_features),
        ("binary", binary_pipeline, binary_features),
        ("categorical", categorical_pipeline, final_categorical_features)
    ]
)

### Continuous, Binary, and Categorical Predictors

The predictors were further divided into three groups:

- continuous numerical variables
- binary clinical-support indicators
- categorical variables

Continuous variables are median-imputed and standardized.

Binary variables are imputed if necessary but are not scaled because they already have a meaningful 0/1 representation.

Categorical variables are imputed and one-hot encoded.

This separation improves interpretability and avoids unnecessary transformation of binary clinical indicators.

## Cross-Validation Strategy

The preprocessing steps and machine-learning model will be combined into a single pipeline.

This is important because imputation, scaling, and one-hot encoding must be fitted separately within each training fold during cross-validation. If preprocessing were performed on the entire training dataset before cross-validation, information from the validation folds could leak into the preprocessing step.

A `StratifiedGroupKFold` strategy will be used so that:

- all ICU stays from the same patient remain within the same fold
- the proportion of prolonged ICU stays remains approximately similar across folds
- model selection and hyperparameter tuning are performed without using the final test set

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
groups_train = clean_df.iloc[train_idx]["subject_id"].reset_index(drop=True)

In [ ]:
print("Training rows:", len(X_train))
print("Training groups:", len(groups_train))
print("Unique training patients:", groups_train.nunique())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

## Baseline Model: Logistic Regression

Logistic regression will be used as the first baseline model.

It is a useful starting point because it is relatively interpretable and provides a benchmark against which more complex models such as Random Forest and XGBoost can be compared.

Performance will be evaluated using group-aware cross-validation so that ICU stays from the same patient do not appear in both the training and validation portions of a fold.

Because the outcome is imbalanced, accuracy alone is not sufficient. We will focus on:

- ROC-AUC
- PR-AUC
- recall
- precision
- F1 score

In [ ]:
from sklearn.model_selection import cross_validate

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

logistic_cv_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [ ]:
import pandas as pd

logistic_cv_summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Recall",
        "Precision",
        "F1"
    ],
    "Mean CV Score": [
        logistic_cv_results["test_roc_auc"].mean(),
        logistic_cv_results["test_pr_auc"].mean(),
        logistic_cv_results["test_recall"].mean(),
        logistic_cv_results["test_precision"].mean(),
        logistic_cv_results["test_f1"].mean()
    ],
    "Std Dev": [
        logistic_cv_results["test_roc_auc"].std(),
        logistic_cv_results["test_pr_auc"].std(),
        logistic_cv_results["test_recall"].std(),
        logistic_cv_results["test_precision"].std(),
        logistic_cv_results["test_f1"].std()
    ]
})

logistic_cv_summary

In [ ]:
logistic_cv_results["test_roc_auc"]

## Random Forest Model

Random Forest will be evaluated as a nonlinear alternative to logistic regression.

Unlike logistic regression, Random Forest can naturally model nonlinear relationships and interactions between predictors without requiring them to be specified in advance.

For example, the effect of renal dysfunction may differ depending on whether a patient is mechanically ventilated or receiving vasoactive support.

The model will be evaluated using the same patient-grouped cross-validation strategy and the same performance metrics used for logistic regression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [ ]:
rf_cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [ ]:
rf_cv_summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Recall",
        "Precision",
        "F1"
    ],
    "Mean CV Score": [
        rf_cv_results["test_roc_auc"].mean(),
        rf_cv_results["test_pr_auc"].mean(),
        rf_cv_results["test_recall"].mean(),
        rf_cv_results["test_precision"].mean(),
        rf_cv_results["test_f1"].mean()
    ],
    "Std Dev": [
        rf_cv_results["test_roc_auc"].std(),
        rf_cv_results["test_pr_auc"].std(),
        rf_cv_results["test_recall"].std(),
        rf_cv_results["test_precision"].std(),
        rf_cv_results["test_f1"].std()
    ]
})

rf_cv_summary

In [ ]:
rf_cv_results["test_roc_auc"]

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [ ]:
xgb_cv_results = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [ ]:
xgb_cv_summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Recall",
        "Precision",
        "F1"
    ],
    "Mean CV Score": [
        xgb_cv_results["test_roc_auc"].mean(),
        xgb_cv_results["test_pr_auc"].mean(),
        xgb_cv_results["test_recall"].mean(),
        xgb_cv_results["test_precision"].mean(),
        xgb_cv_results["test_f1"].mean()
    ],
    "Std Dev": [
        xgb_cv_results["test_roc_auc"].std(),
        xgb_cv_results["test_pr_auc"].std(),
        xgb_cv_results["test_recall"].std(),
        xgb_cv_results["test_precision"].std(),
        xgb_cv_results["test_f1"].std()
    ]
})

xgb_cv_summary

In [ ]:
xgb_cv_results["test_roc_auc"]

## Hyperparameter Optimization

XGBoost was selected as the primary model because it achieved the strongest cross-validated performance among the initial models.

Hyperparameter tuning will be performed using group-aware cross-validation so that all ICU stays from the same patient remain within the same fold.

The main hyperparameters to explore include:

- number of trees
- tree depth
- learning rate
- row subsampling
- feature subsampling

The optimization will focus primarily on ROC-AUC, while PR-AUC and other metrics will also be reviewed.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__learning_rate": [0.02, 0.05, 0.1, 0.15],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
xgb_search.fit(
    X_train,
    y_train,
    groups=groups_train
)

In [ ]:
xgb_search.best_params_

In [ ]:
xgb_search.best_score_

In [ ]:
results = pd.DataFrame(xgb_search.cv_results_)

results[
    [
        "mean_test_score",
        "std_test_score",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__learning_rate",
        "param_model__subsample",
        "param_model__colsample_bytree"
    ]
].sort_values(
    "mean_test_score",
    ascending=False
).head(10)

## Final Evaluation on the Test Set

After model selection and hyperparameter optimization were completed using only the training data, the final tuned XGBoost model was evaluated once on the previously untouched test set.

This provides an estimate of how the finalized model performs on patients who were not used during model development.

The evaluation includes:

- ROC-AUC
- PR-AUC
- accuracy
- recall
- precision
- F1 score
- confusion matrix

The default probability threshold of 0.50 is used for classification metrics. The precision-recall curve is examined descriptively to illustrate the tradeoff between sensitivity and precision, but no classification threshold is optimized using the held-out test set.

In [ ]:
best_xgb_model = xgb_search.best_estimator_

y_test_pred = best_xgb_model.predict(X_test)
y_test_prob = best_xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

test_metrics = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Accuracy",
        "Recall",
        "Precision",
        "F1"
    ],
    "Test Score": [
        roc_auc_score(y_test, y_test_prob),
        average_precision_score(y_test, y_test_prob),
        accuracy_score(y_test, y_test_pred),
        recall_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred),
        f1_score(y_test, y_test_pred)
    ]
})

test_metrics

In [ ]:
confusion_matrix(y_test, y_test_pred)

In [ ]:
print(classification_report(y_test, y_test_pred))

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_test_prob
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(alpha=0.3)

plt.show()

### Precision-Recall Tradeoff

The precision-recall curve demonstrates the expected tradeoff between sensitivity and positive predictive value.

Lower probability thresholds identify a larger proportion of true prolonged ICU stays but generate more false-positive predictions.

Higher thresholds increase precision but miss more patients with prolonged stays.

The default threshold of 0.50 is therefore not inherently optimal. Threshold selection should depend on the intended use of the model and the relative importance of false-negative and false-positive predictions.

## Calibration Analysis

Discrimination metrics such as ROC-AUC and PR-AUC measure how well the model separates higher-risk from lower-risk patients.

Calibration asks a different question:

> When the model predicts a certain probability of prolonged ICU stay, does that probability correspond to the observed event rate?

For example, among patients assigned a predicted risk of approximately 30%, a well-calibrated model should ideally show an observed prolonged-stay rate close to 30%.

Calibration is especially important when model probabilities may eventually be used for clinical risk communication or decision support.

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

prob_true, prob_pred = calibration_curve(
    y_test,
    y_test_prob,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(7, 5))

plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="XGBoost"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed proportion prolonged")
plt.title("Calibration Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    y_test,
    y_test_prob
)

print("Brier score:", brier)

In [ ]:
baseline_probability = y_train.mean()

baseline_predictions = [baseline_probability] * len(y_test)

baseline_brier = brier_score_loss(
    y_test,
    baseline_predictions
)

print("Model Brier score:", brier)
print("Baseline Brier score:", baseline_brier)

### Brier Score Interpretation

The Brier score measures the accuracy of predicted probabilities by comparing each predicted probability with the observed outcome.

Lower values indicate better probabilistic predictions, with a perfect score equal to 0.

The tuned XGBoost model achieved a Brier score of approximately 0.117, compared with approximately 0.158 for a baseline model that predicts the overall event prevalence for every patient.

This indicates that the model provides more informative probability estimates than simply assigning the same baseline risk to all patients.

The Brier score evaluates overall probability accuracy and should be interpreted together with the calibration plot and discrimination metrics such as ROC-AUC and PR-AUC.

## Model Interpretation with SHAP

The final XGBoost model demonstrated useful discrimination and reasonable probability performance, but predictive performance alone does not explain how the model arrives at its predictions.

SHAP (SHapley Additive exPlanations) will be used to estimate the contribution of each predictor to the model's output.

SHAP can help answer two complementary questions:

- **Global interpretation:** Which variables are most influential across the dataset?
- **Local interpretation:** Why did the model assign a particular patient a high or low predicted risk?

SHAP values describe model behavior and association, not causation.

In [ ]:
import shap

In [ ]:
fitted_preprocessor = best_xgb_model.named_steps["preprocessor"]
fitted_xgb = best_xgb_model.named_steps["model"]

X_test_transformed = fitted_preprocessor.transform(X_test)

In [ ]:
feature_names = fitted_preprocessor.get_feature_names_out()

feature_names[:20], len(feature_names)

In [ ]:
explainer = shap.TreeExplainer(fitted_xgb)

shap_values = explainer(X_test_transformed)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names,
    max_display=20
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names,
    plot_type="bar",
    max_display=20
)

## Save Final Model

The finalized machine-learning pipeline includes both preprocessing and the tuned XGBoost classifier.

Saving the complete pipeline ensures that future predictions use exactly the same:

- missing-value imputation
- numerical scaling
- categorical encoding
- trained XGBoost model

This is especially important for deployment because the web application should receive raw clinical inputs and apply the same transformations used during model development.

In [ ]:
import joblib

joblib.dump(
    best_xgb_model,
    "../models/prolonged_icu_xgb_pipeline.pkl"
)

In [ ]:
import os

model_path = "../models/prolonged_icu_xgb_pipeline.pkl"

print("Model exists:", os.path.exists(model_path))
print("Model size (MB):", round(os.path.getsize(model_path) / 1024**2, 2))

In [ ]:
loaded_model = joblib.load(
    "../models/prolonged_icu_xgb_pipeline.pkl"
)

loaded_prob = loaded_model.predict_proba(
    X_test.iloc[:5]
)[:, 1]

loaded_prob

In [ ]:
original_prob = best_xgb_model.predict_proba(
    X_test.iloc[:5]
)[:, 1]

print("Original:", original_prob)
print("Reloaded:", loaded_prob)